# 04. Part-to-Whole Analysis: How Does Each Part Contribute?
### Primary Analytical Purpose: *How does each part contribute to the total?*
**Lecture Reference:** Purpose 4 — Part-to-Whole (Slides 21–24, 45)

Part-to-whole analysis evaluates how individual segments combine to form a complete 100% total. It addresses questions such as:
- What share of the total does each component represent?
- How has the composition of the total shifted over time?
- Which segment is growing its market share, and which is shrinking?

---

## 1. Scenario and Dataset Preview
**Business Scenario:** Qatar Energy and Kahramaa monitor the quarterly electricity generation mix (in Gigawatt-hours, GWh) across 2024. The generation mix includes three primary sources: *Natural Gas*, *Al Kharsaah Solar*, and *Cogeneration*. The goal is to track progress toward national clean energy targets while meeting peak summer power demand.

The dataset is loaded from `data/qatar_energy_generation.csv`. Let us inspect the data.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

# Set modern Seaborn theme
sns.set_theme(style="whitegrid")

# Load electricity generation dataset
df = pd.read_csv("data/qatar_energy_generation.csv")

# Display the dataset
print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
df

## 2. Primary Visualization: Stacked Bar Chart (The Default)
- **What the chart shows:** Total generation (bar height) alongside the contribution of each source (stacked segments) for each quarter.
- **When it is useful:** When absolute totals and composition both matter across multiple periods (Slide 21).
- **Library note (Slide 24):** Seaborn has no native stacked-bar API, so we use Pandas `.plot(kind='bar', stacked=True)` on a pivoted wide DataFrame. Plotly Express natively accepts tidy (long) data.

In [ ]:
# Pivot data to wide format for Pandas / Matplotlib stacked bar
wide_df = df.pivot(index="quarter", columns="energy_source", values="generation_gwh")
# Reorder columns: baseline gas first, solar second, cogen third
wide_df = wide_df[["Natural Gas", "Al Kharsaah Solar", "Cogeneration"]]

# --- Seaborn / Pandas Implementation: Stacked Bar Chart ---
colors = ["#2B5C8F", "#E69F00", "#14A098"]  # Distinct palette for 3 fuels
ax = wide_df.plot(kind="bar", stacked=True, color=colors, figsize=(8.5, 4.5), rot=0)

ax.set_ylim(0, 19000)
ax.set_title("Qatar Electricity Generation Peaked at 17,000 GWh in Q3 2024", fontsize=13, weight="bold")
ax.set_xlabel("Quarter", fontsize=11)
ax.set_ylabel("Electricity Generation (GWh)", fontsize=11)
ax.legend(title="Energy Source", frameon=True)

plt.tight_layout()
plt.show()

In [ ]:
# --- Plotly Express Implementation: Interactive Stacked Bar Chart ---
fig = px.bar(
    df, 
    x="quarter", 
    y="generation_gwh", 
    color="energy_source",
    color_discrete_map={"Natural Gas": "#2B5C8F", "Al Kharsaah Solar": "#E69F00", "Cogeneration": "#14A098"},
    title="Qatar Electricity Generation Peaked at 17,000 GWh in Q3 2024",
    labels={"quarter": "Quarter", "generation_gwh": "Generation (GWh)", "energy_source": "Energy Source"}
)

fig.update_layout(yaxis_title="Generation (GWh)")
fig.show()

### Structured Interpretation: The 5-Step Framework
1. **Question:** How does overall electricity demand fluctuate across quarters, and how much power does solar contribute?
2. **Visualization:** Stacked bar chart showing absolute GWh generation per quarter by source.
3. **Observation:** Total power generation peaks dramatically in Q3 (17,000 GWh) due to summer air conditioning, compared to 10,880 GWh in Q1. Natural gas accounts for the vast majority of output, while Al Kharsaah Solar output increased from 720 GWh in Q1 to 1,480 GWh in Q4.
4. **Insight:** Natural gas provides critical peaking capacity to handle summer loads; solar output doubled over the year as expansion phases connected to the grid.
5. **Implication:** Continue solar deployment to support daytime cooling peaks and conserve gas for export.

## 3. Alternative 1: 100% Stacked Bar Chart (Pure Share Mix)
- **What the chart shows:** Relative percentage share of each energy source, normalized so each bar equals 100%.
- **When to use instead:** When only the share mix matters and total volumes differ wildly between periods (Slide 21 & 45).
- **Critical rule from lecture (Slide 22 & 23):** **Only the bottom band has a stable baseline.** Put the series you most want tracked at the bottom on the stable baseline!

In [ ]:
# Normalize wide data to 100% percentages
share_df = wide_df.div(wide_df.sum(axis=1), axis=0) * 100

# Put Solar at the bottom to give it the stable zero baseline!
share_ordered = share_df[["Al Kharsaah Solar", "Natural Gas", "Cogeneration"]]

# --- Pandas / Seaborn Implementation: 100% Stacked Bar ---
plt.figure(figsize=(8.5, 4.5))
ax_100 = share_ordered.plot(
    kind="bar", 
    stacked=True, 
    color=["#E69F00", "#2B5C8F", "#14A098"], 
    figsize=(8.5, 4.5), 
    rot=0
)

ax_100.set_ylim(0, 100)
ax_100.set_title("Al Kharsaah Solar Share Rose from 6.6% in Q1 to 12.1% in Q4", fontsize=13, weight="bold")
ax_100.set_xlabel("Quarter", fontsize=11)
ax_100.set_ylabel("Share of Generation (%)", fontsize=11)
ax_100.legend(title="Energy Source", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.show()

In [ ]:
# --- Plotly Express: 100% Stacked Bar using barnorm='percent' ---
fig_norm = px.bar(
    df, 
    x="quarter", 
    y="generation_gwh", 
    color="energy_source",
    barnorm="percent",
    color_discrete_map={"Natural Gas": "#2B5C8F", "Al Kharsaah Solar": "#E69F00", "Cogeneration": "#14A098"},
    title="Al Kharsaah Solar Share Rose from 6.6% in Q1 to 12.1% in Q4",
    labels={"quarter": "Quarter", "generation_gwh": "Share (%)", "energy_source": "Source"}
)

fig_norm.update_layout(yaxis_title="Share of Generation (%)")
fig_norm.show()

### Interpretation for 100% Stacked Bar
- **Observation:** Solar's share (bottom amber band) increased steadily from 6.6% in Q1 to 12.1% in Q4. Natural gas share decreased from 90.5% in Q1 to 85.2% in Q4.
- **Derived Insight:** Solar's proportion doubled in 2024, demonstrating tangible progress in Qatar's renewable energy transition.

## 4. Alternative 2: Donut / Pie Chart (Single Snapshot Only)
- **What the chart shows:** Proportions of a whole at a single snapshot in time.
- **When to use instead (Slide 21 & 23):** Only with 2–4 parts and large differences.
- **Strict Guidelines from Lecture:**
  - Angle and area are read poorly by human vision; **label every slice with its percentage**.
  - **Never use 3D or exploded pies**, which distort area geometry.
  - **Never use two pies side by side to compare change** — use a stacked bar chart instead!

In [ ]:
# Aggregate annual totals across 2024
annual_totals = df.groupby("energy_source", as_index=False)["generation_gwh"].sum()

# --- Plotly Express: Clean Donut Chart ---
fig_donut = px.pie(
    annual_totals, 
    values="generation_gwh", 
    names="energy_source",
    hole=0.45,  # Creates clean donut cutout
    color="energy_source",
    color_discrete_map={"Natural Gas": "#2B5C8F", "Al Kharsaah Solar": "#E69F00", "Cogeneration": "#14A098"},
    title="2024 Annual Electricity Mix: Natural Gas Contributed 88.6%"
)

# Show direct percentage and label on each slice
fig_donut.update_traces(textposition="inside", textinfo="percent+label")
fig_donut.show()

### Interpretation for Donut Chart
- **Observation:** Across full-year 2024, Natural Gas provided 88.6% of power, Al Kharsaah Solar contributed 8.8%, and Cogeneration provided 2.7%.
- **Derived Insight:** While solar grew rapidly to reach 12.1% in Q4, natural gas remains the overwhelming annual foundation of grid stability.

## 5. Best Practices & Common Mistakes (Lecture Principles)
**Visualization Best Practices (Slide 23 & 46):**
- **Only use part-to-whole when parts genuinely sum to a meaningful total** (100%).
- **Put the series you most want tracked at the bottom** on the stable baseline.
- **Keep to 3–5 segments;** group any smaller segments into 'Other'.
- **Label shares directly as percentages.**

**Common Mistakes & Misleading Designs:**
- **Pies with 10 slices and a remote legend:** Reader must decode colors back and forth.
- **Stacking non-additive categories:** Stacking metrics that overlap or do not sum to a true whole.
- **Confusing a falling share with a falling amount (Slide 22 & 23):** Natural gas *share* fell in Q4, but its absolute volume remained high. Share and amount are distinct claims!
- **Side-by-side pies:** Impossible to judge whether a slice grew or shrank; stacked bars solve this.

## 6. Hands-on Practice Exercises

### Exercise 1: Chart Interpretation
In Q3, Natural Gas generated 15,200 GWh, while in Q1 it generated 9,850 GWh. Yet its percentage share in Q3 was 89.4% vs 90.5% in Q1.
1. **Observation:** Why did gas generation volume increase by +54% while its share dropped by -1.1 percentage points?
2. **Insight:** How does total demand impact percentage share calculations?
3. **Implication:** Why should energy analysts present both absolute GWh and percentage share?

### Exercise 2: Code Modification
Re-plot the 100% stacked bar chart, but place `Cogeneration` at the bottom baseline instead of Solar.

In [ ]:
# Exercise 2: Modify the column order to place Cogeneration at the bottom
# share_cogen_first = share_df[['Cogeneration', 'Natural Gas', 'Al Kharsaah Solar']]
# ...

### Exercise 3: Single Period Pie Chart
Filter `df` to Q4 only. Create a pie chart with Seaborn / Matplotlib `plt.pie()` with direct percentage labels.

In [ ]:
# Exercise 3: Complete the Q4 pie chart below
# q4_df = df[df['quarter'] == 'Q4']
# plt.figure(figsize=(5, 5))
# plt.pie(q4_df['generation_gwh'], labels=q4_df['energy_source'], autopct='%1.1f%%')
# plt.show()

## 7. Key Takeaways
1. **Stacked bar charts show both total magnitude and composition;** 100% stacked bars isolate changing percentage mix.
2. **Only the bottom band has a stable zero baseline:** Place the primary series of interest at the bottom.
3. **Never confuse a falling share with a falling amount:** A category can grow in absolute volume while shrinking in relative share.
4. **Use pie/donut charts sparingly:** Maximum 2–4 slices, single moment in time, with direct percentage labels.